# Figure 3a: Individual I-Z Traces
Comparison of individual I-Z traces between contaminated and UHV cleaved graphite at 250 pA

In [1]:
import os
import numpy as np
import holoviews as hv
import rhkpy
import requests

hv.extension('bokeh')

In [2]:
import sys
print(f'Python version: {sys.version}')
print(f'numpy version: {np.__version__}')
print(f'holoviews version: {hv.__version__}')
print(f'rhkpy version: {rhkpy.__version__}')

Python version: 3.13.5 | packaged by Anaconda, Inc. | (main, Jun 12 2025, 11:23:37) [Clang 14.0.6 ]
numpy version: 2.3.1
holoviews version: 1.21.0
rhkpy version: 1.3.9


In [3]:
# Download source files from Zenodo if missing
ZENODO_FILES = {
    'iz_c250.sm4': 'https://zenodo.org/records/17469441/files/I-Z_Kalvin_HOPG_UHV_exf240717_9K_2024_07_22_16_00_20_383.sm4?download=1',
    'iz_map_in4.sm4': 'https://zenodo.org/records/17469441/files/MK_ABC_FLG_25_9K_2023_11_08_06_52_20_288.sm4?download=1',
}

for local, url in ZENODO_FILES.items():
    if url is None or os.path.exists(local):
        continue
    print(f'Downloading {local}...')
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(local, 'wb') as fh:
        fh.write(response.content)
    print(f'✓ {local} downloaded')

In [4]:
# Load and process contaminated graphite data (250 pA)
iz_map_in4 = rhkpy.rhkdata('iz_map_in4.sm4')
iz_in4_up = iz_map_in4.spectra.current.sel(zscandir='up')
iz_in4_up['z'] = iz_in4_up.z[::-1].data

# Stack all spectra dimensions
iz_in4_up = iz_in4_up.stack(allspec=('repetitions', 'specpos_x', 'specpos_y'))
iz_in4_up = iz_in4_up.drop_vars(['specpos_x', 'specpos_y', 'repetitions'])
iz_in4_up = iz_in4_up.assign_coords(allspec=iz_in4_up['allspec'])
iz_in4_up['z'] = iz_in4_up['z'] * 10
iz_in4_up['z'] = iz_in4_up.z[::-1].data
iz_in4_up.data = np.abs(iz_in4_up.data)

# Load UHV cleaved reference data
iz_c250 = rhkpy.rhkdata('iz_c250.sm4')
iz_c250.spectra['z'] = iz_c250.spectra['z'] * 10

In [5]:
# Crop and mask data to 0-10 Å range and max 250 pA
iz_in4_cropped = iz_in4_up.sel(z=slice(0, 10)).where(iz_in4_up.sel(z=slice(0, 10)) <= 250)
iz_c250_cropped = iz_c250.spectra.current.sel(zscandir='up').sel(z=slice(0, 10)).where(
    iz_c250.spectra.current.sel(zscandir='up').sel(z=slice(0, 10)) <= 250
)

In [6]:
# Plot first 20 individual traces from each dataset
plot_iz = iz_in4_cropped[:, 0].hvplot.line(x='z', label='contaminated').opts(
    alpha=0.2, line_width=0.6, color='red'
)
plot_iz *= iz_c250_cropped[:, 0].hvplot.line(x='z', label='UHV cleaved').opts(
    alpha=0.2, line_width=0.6, color='blue'
)

for i in range(1, 20):
    plot_iz *= iz_in4_cropped[:, i].hvplot.line(x='z').opts(
        alpha=0.2, line_width=0.6, color='red'
    )
    plot_iz *= iz_c250_cropped[:, i].hvplot.line(x='z').opts(
        alpha=0.2, line_width=0.6, color='blue'
    )

plot_iz_traces = plot_iz.opts(
    xlim=(0, 10),
    ylim=(0, 250),
    width=700,
    height=580,
    xlabel='tip - sample distance (Å)',
    ylabel='current (pA)',
    title='Individual I-Z Traces (250 pA)',
    show_grid=True,
    legend_position='right'
)
plot_iz_traces

:Overlay
   .Curve.Contaminated :Curve   [z]   (current)
   .Curve.UHV_cleaved  :Curve   [z]   (current)
   .Curve.I            :Curve   [z]   (current)
   .Curve.II           :Curve   [z]   (current)
   .Curve.III          :Curve   [z]   (current)
   .Curve.IV           :Curve   [z]   (current)
   .Curve.V            :Curve   [z]   (current)
   .Curve.VI           :Curve   [z]   (current)
   .Curve.VII          :Curve   [z]   (current)
   .Curve.VIII         :Curve   [z]   (current)
   .Curve.IX           :Curve   [z]   (current)
   .Curve.X            :Curve   [z]   (current)
   .Curve.XI           :Curve   [z]   (current)
   .Curve.XII          :Curve   [z]   (current)
   .Curve.XIII         :Curve   [z]   (current)
   .Curve.XIV          :Curve   [z]   (current)
   .Curve.XV           :Curve   [z]   (current)
   .Curve.XVI          :Curve   [z]   (current)
   .Curve.XVII         :Curve   [z]   (current)
   .Curve.XVIII        :Curve   [z]   (current)
   .Curve.XIX          :Curve   [z]   (current)
   .Curve.XX           :Curve   [z]   (current)
   .Curve.XXI          :Curve   [z]   (current)
   .Curve.XXII         :Curve   [z]   (current)
   .Curve.XXIII        :Curve   [z]   (current)
   .Curve.XXIV         :Curve   [z]   (current)
   .Curve.XXV          :Curve   [z]   (current)
   .Curve.XXVI         :Curve   [z]   (current)
   .Curve.XXVII        :Curve   [z]   (current)
   .Curve.XXVIII       :Curve   [z]   (current)
   .Curve.XXIX         :Curve   [z]   (current)
   .Curve.XXX          :Curve   [z]   (current)
   .Curve.XXXI         :Curve   [z]   (current)
   .Curve.XXXII        :Curve   [z]   (current)
   .Curve.XXXIII       :Curve   [z]   (current)
   .Curve.XXXIV        :Curve   [z]   (current)
   .Curve.XXXV         :Curve   [z]   (current)
   .Curve.XXXVI        :Curve   [z]   (current)
   .Curve.XXXVII       :Curve   [z]   (current)
   .Curve.XXXVIII      :Curve   [z]   (current)

In [7]:
# hv.save(plot_iz_traces, 'fig3a_iz_traces.html')